In [1]:
rm(list = ls())

# check and install CRAN packages
cran_packages <- c("ggplot2", "ape", "dplyr", "tibble", "tidyverse")

for (pkg in cran_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
  }
}

lapply(cran_packages, library, character.only = TRUE)

# check and install Bioconductor packages  
bioc_packages <- c("ggtree", "ggtreeExtra", "ggnewscale")

if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}

for (pkg in bioc_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    BiocManager::install(pkg)
  }
}

lapply(bioc_packages, library, character.only = TRUE)

# this funtion is needed
is.waive <- function(x) inherits(x, "waiver")



Attaching package: ‘dplyr’


The following object is masked from ‘package:ape’:

    where


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


── Attaching core tidyverse packages ─────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.1     ✔ readr     2.1.6
✔ lubridate 1.9.4     ✔ stringr   1.6.0
✔ purrr     1.2.0     ✔ tidyr     1.3.2
── Conflicts ───────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
✖ dplyr::where()  masks ape::where()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


[[1]]
[1] "ggplot2"   "repr"      "stats"     "graphics"  "grDevices" "utils"    
[7] "datasets"  "methods"   "base"     

[[2]]
 [1] "ape"       "ggplot2"   "repr"      "stats"     "graphics"  "grDevices"
 [7] "utils"     "datasets"  "methods"   "base"     

[[3]]
 [1] "dplyr"     "ape"       "ggplot2"   "repr"      "stats"     "graphics" 
 [7] "grDevices" "utils"     "datasets"  "methods"   "base"     

[[4]]
 [1] "tibble"    "dplyr"     "ape"       "ggplot2"   "repr"      "stats"    
 [7] "graphics"  "grDevices" "utils"     "datasets"  "methods"   "base"     

[[5]]
 [1] "lubridate" "forcats"   "stringr"   "purrr"     "readr"     "tidyr"    
 [7] "tidyverse" "tibble"    "dplyr"     "ape"       "ggplot2"   "repr"     
[13] "stats"     "graphics"  "grDevices" "utils"     "datasets"  "methods"  
[19] "base"

ggtree v3.16.3 Learn more at https://yulab-smu.top/contribution-tree-data/

Please cite:

Guangchuang Yu. Using ggtree to visualize data on tree-like structures.
Current Protocols in Bioinformatics. 2020, 69:e96. doi:10.1002/cpbi.96


Attaching package: ‘ggtree’


The following object is masked from ‘package:tidyr’:

    expand


The following object is masked from ‘package:ape’:

    rotate


ggtreeExtra v1.18.1 For help: https://yulab-smu.top/treedata-book/

If you use the ggtree package suite in published research, please cite
the appropriate paper(s):

S Xu, Z Dai, P Guo, X Fu, S Liu, L Zhou, W Tang, T Feng, M Chen, L
Zhan, T Wu, E Hu, Y Jiang, X Bo, G Yu. ggtreeExtra: Compact
visualization of richly annotated phylogenetic data. Molecular Biology
and Evolution. 2021, 38(9):4039-4042. doi: 10.1093/molbev/msab166

Guangchuang Yu, Tommy Tsan-Yuk Lam, Huachen Zhu, Yi Guan. Two methods
for mapping and visualizing associated data on phylogeny using ggtree.
Molecular Biology and Evolution

[[1]]
 [1] "ggtree"    "lubridate" "forcats"   "stringr"   "purrr"     "readr"    
 [7] "tidyr"     "tidyverse" "tibble"    "dplyr"     "ape"       "ggplot2"  
[13] "repr"      "stats"     "graphics"  "grDevices" "utils"     "datasets" 
[19] "methods"   "base"     

[[2]]
 [1] "ggtreeExtra" "ggtree"      "lubridate"   "forcats"     "stringr"    
 [6] "purrr"       "readr"       "tidyr"       "tidyverse"   "tibble"     
[11] "dplyr"       "ape"         "ggplot2"     "repr"        "stats"      
[16] "graphics"    "grDevices"   "utils"       "datasets"    "methods"    
[21] "base"       

[[3]]
 [1] "ggnewscale"  "ggtreeExtra" "ggtree"      "lubridate"   "forcats"    
 [6] "stringr"     "purrr"       "readr"       "tidyr"       "tidyverse"  
[11] "tibble"      "dplyr"       "ape"         "ggplot2"     "repr"       
[16] "stats"       "graphics"    "grDevices"   "utils"       "datasets"   
[21] "methods"     "base"

In [2]:
keep_tip_order <- function(df, tip_labels, id_col = "ID") {
  df %>%
    filter(.data[[id_col]] %in% tip_labels) %>%
    mutate(!!id_col := factor(.data[[id_col]], levels = tip_labels)) %>%
    arrange(.data[[id_col]])
}

add_fruit_tile <- function(p, df, y = ID, x, fill, offset, width, values, legend_order) {
  p +
    new_scale_fill() +
    geom_fruit(
      data = df,
      geom = geom_tile,
      mapping = aes(y = {{ y }}, x = {{ x }}, fill = {{ fill }}),
      color = "white",
      offset = offset,
      width = width
    ) +
    scale_fill_manual(
      values = values,
      guide = guide_legend(keywidth = 2, keyheight = 2, order = legend_order)
    )
}

## Load tree

In [3]:
tree <- read.tree("./lib/ppr.fa.trimal.phy.contree")
tip_labels <-  tree$tip.label

p <- ggtree(tree, layout = "fan", color = "black", size = 1.5) +
  geom_treescale(x = 0, y = 45, width = 0.7) +
  geom_tiplab2(
    offset = 0.08, size = 0, color = "grey",
    align = TRUE, linetype = NA, linesize = 0.1
  )

p <- open_tree(p, 8)


Warning message:
“`aes_()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`
ℹ The deprecated feature was likely used in the ggtree package.
  Please report the issue at <https://github.com/YuLab-SMU/ggtree/issues>.”
Warning message in fortify(data, ...):
“Arguments in `...` must be used.
✖ Problematic arguments:
• as.Date = as.Date
• yscale_mapping = yscale_mapping
• hang = hang
• color = "black"
• size = 1.5
ℹ Did you misspell an argument name?”
Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.
ℹ The deprecated feature was likely used in the ggtree package.
  Please report the issue at <https://github.com/YuLab-SMU/ggtree/issues>.”
Scale for y is already present.
Adding another scale for y, which will replace the existing scale.
Warning message:
“`aes_string()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`.
ℹ See also `vignette("ggplot2-in-packages")

## Add chr information

In [4]:
chr_data <- read.delim("./lib/gma_ppr_chr.txt", sep = "\t", header = TRUE, check.names = FALSE) %>%
  transmute(
    ID,
    Count,
    Chromosome = case_when(
      Chromosome == "9"  ~ "Chr9",
      Chromosome == "16" ~ "Chr16",
      TRUE               ~ "Other"
    ),
    Chromosome = factor(Chromosome, levels = c("Chr9", "Chr16", "Other"))
  ) %>%
  keep_tip_order(tip_labels)

p <- add_fruit_tile(
  p, chr_data,
  x = Count, fill = Chromosome,
  offset = -0.16, width = 0.3,
  values = c("Chr16" = "#9E2214", "Chr9" = "#4363d8", "Other" = "#EDE9D5"),
  legend_order = 3
)



## Add trigger number information

In [5]:
miRNA_target <- read.delim("./lib/gma_trigger_number.txt", sep = "\t", header = TRUE, check.names = FALSE) %>%
  transmute(
    ID,
    Trigger = 1,
    Number = as.numeric(Number),
    Number = cut(
      Number,
      breaks = c(0, 1, 3, 5, 7),
      labels = c("0", "1-3", "3-5", "5-7"),
      include.lowest = TRUE
    )
  ) %>%
  keep_tip_order(tip_labels)

p <- add_fruit_tile(
  p, miRNA_target,
  x = Trigger, fill = Number,
  offset = -0.14, width = 0.3,
  values = c("0" = "#FDF6E7", "1-3" = "#F5CFC8", "3-5" = "#EB99B2", "5-7" = "#C0408C"),
  legend_order = 4
)


## Add sRNA abundance information

In [6]:
sRNA_data <- read.delim(
  "./lib/gma_ppr_tpm_srna.txt",
  sep = "\t", header = TRUE, row.names = 1, check.names = FALSE
)

sRNA_temp <- tibble(
  ID = rownames(sRNA_data),
  LeafTPM = log2(as.numeric(sRNA_data$Leaf) + 1)
) %>%
  mutate(
    LeafTPM = replace_na(LeafTPM, 0),
    Abundance = cut(
      LeafTPM,
      breaks = c(0, 1, 3, 5, 9),
      labels = c("0", "1-3", "3-5", "5-9"),
      include.lowest = TRUE
    ),
    Leaf = 1   
  ) %>%
  select(ID, Leaf, Abundance) %>%
  keep_tip_order(tip_labels)

p <- add_fruit_tile(
  p, sRNA_temp,
  x = Leaf, fill = Abundance,
  offset = -0.14, width = 0.3,
  values = c("0" = "#F7E9C6", "1-3" = "#F1D078", "3-5" = "#CE9B36", "5-9" = "#644B15"),
  legend_order = 5
)


In [7]:
ggsave(p, filename = "gma_ppr.fa.trimal.phy.contree.pdf", width = 80, height = 80, units = "cm", limitsize = FALSE)